# Fresh K=3 experiment — Stage 1: kline dataset generation
Run every cell in order. This notebook stops after generating, verifying, and archiving the 30-candle image corpus. Do not run clustering yet.

In [ ]:
# Install the renderer versions used by the byte-for-byte regression test.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib==3.10.8', 'pandas==2.2.3', 'Pillow==12.2.0'], check=True)


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile
import matplotlib, pandas as pd, PIL

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
CSV_NAME = '0bd0af71-8704-4113-8136-b01a2ac5d1d1.csv'
EXPECTED_CSV_SHA = 'c938439e45cce76aee7234c62be1ba5aaa6b80c6f3598924715d5d0c9563ce8d'
EXPECTED_FIRST_IMAGE_SHA = '9551df8b81bd0b63ff071884c446525942e20e7856c70c198e3c423aabe80487'

def unique_input(name):
    matches = list(INPUT.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {name!r} under /kaggle/input; found {matches}')
    return matches[0]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

csv_path = unique_input(CSV_NAME)
generator = unique_input('kline_dataset_generator.py')
preflight = unique_input('preflight_csv.py')
verifier = unique_input('verify_generated_dataset.py')
template = unique_input('kline_dataset_config_kaggle.json')
reference = unique_input('sample_0_29.jpg')
print('CSV:', csv_path)
print('Generator:', generator)
print('Python:', sys.version)
print('matplotlib:', matplotlib.__version__, 'pandas:', pd.__version__, 'Pillow:', PIL.__version__)
assert sha256(csv_path) == EXPECTED_CSV_SHA, 'Wrong or modified source CSV'


## 1. Audit the complete CSV
This must pass before any JPEG is generated. Missing hourly candles are documented and preserved; they are not imputed.

In [ ]:
audit_path = WORK / 'source_csv_audit.json'
subprocess.run([sys.executable, str(preflight), str(csv_path), '--expected-sha256', EXPECTED_CSV_SHA, '--output', str(audit_path)], check=True)
audit = json.loads(audit_path.read_text())
assert audit['rows'] == 48281
assert audit['candidate_windows'] == 48252
assert audit['valid_windows'] == 47817
assert audit['skipped_windows'] == 435
assert audit['invalid_ohlc_rows'] == 0
assert audit['duplicate_timestamps'] == 0
print('CSV PREFLIGHT PASSED')


## 2. Create the executed runtime configuration and run a smoke test
The smoke test renders 291 rolling images from the first 320 CSV rows. Its first JPEG must exactly match the historical reference.

In [ ]:
runtime_config = json.loads(template.read_text())
runtime_config['ohlc_csv'] = str(csv_path)
runtime_config['output_dir'] = str(WORK / '01_kline_dataset_k3')
runtime_config_path = WORK / 'kline_dataset_config_executed.json'
runtime_config_path.write_text(json.dumps(runtime_config, indent=2) + '\n')

smoke_root = WORK / '_smoke_kline_generation'
if smoke_root.exists():
    shutil.rmtree(smoke_root)
smoke_root.mkdir(parents=True)
smoke_csv = smoke_root / 'first_320_rows.csv'
pd.read_csv(csv_path, nrows=320).to_csv(smoke_csv, index=False)
smoke_config = dict(runtime_config)
smoke_config['ohlc_csv'] = str(smoke_csv)
smoke_config['output_dir'] = str(smoke_root / 'output')
smoke_config['existing_output_policy'] = 'error'
smoke_config['split'] = {'validation_start_index': None, 'train_fraction': 0.9, 'embargo_candles': 30}
smoke_config_path = smoke_root / 'config.json'
smoke_config_path.write_text(json.dumps(smoke_config, indent=2) + '\n')
subprocess.run([sys.executable, str(generator), '--config', str(smoke_config_path)], check=True)
smoke_first = smoke_root / 'output/images/sample_0_29.jpg'
assert sha256(smoke_first) == EXPECTED_FIRST_IMAGE_SHA
assert smoke_first.read_bytes() == reference.read_bytes()
assert len(list((smoke_root / 'output/images').glob('*.jpg'))) == 291
print('RENDERER SMOKE TEST PASSED — first JPEG is byte-identical')


## 3. Generate the complete corpus
This is CPU and disk-I/O work. It may take a long time even if the Notebook accelerator is set to T4 x2. Keep the notebook tab open and preserve all console output.

In [ ]:
output_root = Path(runtime_config['output_dir'])
subprocess.run([sys.executable, str(generator), '--config', str(runtime_config_path)], check=True)
print('FULL GENERATION FINISHED:', output_root)


## 4. Strict verification and provenance capture
Every JPEG is opened and checked. The manifest counts, embargo, skipped windows, source hash, and first-image hash are locked.

In [ ]:
subprocess.run([sys.executable, str(verifier), '--dataset', str(output_root), '--source-csv', str(csv_path), '--all-image-dimensions'], check=True)
provenance = output_root / 'provenance'
provenance.mkdir(exist_ok=True)
for source in [generator, preflight, verifier, runtime_config_path, audit_path]:
    shutil.copy2(source, provenance / source.name)
environment = {
    'python': sys.version, 'platform': platform.platform(),
    'matplotlib': matplotlib.__version__, 'pandas': pd.__version__, 'Pillow': PIL.__version__,
    'source_csv': str(csv_path), 'source_csv_sha256': sha256(csv_path),
}
(provenance / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
print('STRICT VERIFICATION PASSED')


## 5. Create the download archive
JPEGs are stored without recompression to make archiving faster. Download the ZIP unchanged and send it for audit before clustering.

In [ ]:
archive = WORK / '01_kline_dataset_k3.zip'
if archive.exists():
    archive.unlink()
files = sorted(path for path in output_root.rglob('*') if path.is_file())
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    for number, path in enumerate(files, 1):
        zf.write(path, Path(output_root.name) / path.relative_to(output_root))
        if number % 5000 == 0:
            print(f'Archived {number:,}/{len(files):,} files')
archive_hash = sha256(archive)
hash_file = WORK / '01_kline_dataset_k3.zip.sha256.txt'
hash_file.write_text(f'{archive_hash}  {archive.name}\n')
print('READY TO DOWNLOAD:', archive)
print('Archive bytes:', archive.stat().st_size)
print('Archive SHA-256:', archive_hash)
print('STOP HERE. DO NOT RUN CLUSTERING YET.')
